# Excel Ingestion Demo — Product Sales by Region Dataset

## Flow

```text
Excel file
↓
Load workbook
↓
Detect sheet names
↓
Read selected sheet
↓
Clean column names
↓
Validate required fields
↓
Separate required-field missing values and optional-field missing values
↓
Remove duplicate rows
↓
Save raw output
↓
Save staging output
↓
Save clean output
↓
Generate ingestion log
```

## Expected Input

```text
data/sample_inputs/Product-Sales-Region.xlsx
```

## Expected Outputs

```text
data/raw/excel/product_sales_region_raw.xlsx
data/staging/excel/product_sales_region_staging.csv
data/clean/excel/product_sales_region_clean.csv
logs/excel_ingestion_log.json
```

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import json
import uuid
import re
import shutil

from pathlib import Path
from datetime import datetime, timezone

## 2. Define Project Paths

In [ ]:
def find_project_root(current_path: Path) -> Path:
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError("Project root not found. Make sure data/ folder exists.")


def to_relative_path(path: Path, project_root: Path) -> str:
    return path.resolve().relative_to(project_root.resolve()).as_posix()


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

input_path = PROJECT_ROOT / "data" / "sample_inputs" / "Product-Sales-Region.xlsx"

raw_dir = PROJECT_ROOT / "data" / "raw" / "excel"
staging_dir = PROJECT_ROOT / "data" / "staging" / "excel"
clean_dir = PROJECT_ROOT / "data" / "clean" / "excel"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "product_sales_region_raw.xlsx"
staging_output_path = staging_dir / "product_sales_region_staging.csv"
clean_output_path = clean_dir / "product_sales_region_clean.csv"
log_output_path = log_dir / "excel_ingestion_log.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Raw output:", raw_output_path)
print("Staging output:", staging_output_path)
print("Clean output:", clean_output_path)
print("Log output:", log_output_path)

Current dir: f:\data\new\quanskill\DataVision_Duy\week2\notebooks\data_team
Project root: F:\data\new\quanskill\DataVision_Duy\week2
Input path: F:\data\new\quanskill\DataVision_Duy\week2\data\sample_inputs\Product-Sales-Region.xlsx
Input exists: True
Raw output: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\excel\product_sales_region_raw.xlsx
Staging output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\excel\product_sales_region_staging.csv
Clean output: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\excel\product_sales_region_clean.csv
Log output: F:\data\new\quanskill\DataVision_Duy\week2\logs\excel_ingestion_log.json


## 3. Validate Input File

In [ ]:
if not input_path.exists():
    raise FileNotFoundError(f"Excel file not found: {input_path}")

if input_path.stat().st_size == 0:
    raise ValueError("Excel file is empty.")

if input_path.suffix.lower() not in [".xlsx", ".xls"]:
    raise ValueError("Input file is not an Excel file.")

print("Input validation passed.")

Input validation passed.


## 4. Helper Function

In [ ]:
def clean_column_name(column_name: str) -> str:
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    return column_name.strip("_")

## 5. Start Ingestion Run

In [ ]:
run_id = str(uuid.uuid4())
source_name = "product_sales_region_excel"
source_type = "excel"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: 5edff1eb-857e-4515-8a5d-31e2628466dc
Start time: 2026-06-14T05:44:52.398162+00:00


## 6. Load Workbook and Detect Sheet Names

In [ ]:
try:
    excel_file = pd.ExcelFile(input_path)
    sheet_names = excel_file.sheet_names

    status = "success"
    error_message = None

    print("Workbook loaded successfully.")
    print("Sheet names:", sheet_names)

except Exception as error:
    status = "failed"
    error_message = str(error)
    raise

Workbook loaded successfully.
Sheet names: ['Sheet1']


## 7. Select Sheet

In [ ]:
selected_sheet = sheet_names[0]

print("Selected sheet:", selected_sheet)

Selected sheet: Sheet1


## 8. Read Selected Sheet

In [ ]:
df_raw = pd.read_excel(
    input_path,
    sheet_name=selected_sheet
)

records_read = len(df_raw)

print("Excel sheet loaded successfully.")
print("Records read:", records_read)
print("Column count:", len(df_raw.columns))
print("Columns:", df_raw.columns.tolist())

df_raw.head(5)

Excel sheet loaded successfully.
Records read: 1500
Column count: 19
Columns: ['Date', 'Region', 'Product', 'Quantity', 'UnitPrice', 'StoreLocation', 'CustomerType', 'Discount', 'Salesperson', 'TotalPrice', 'PaymentMethod', 'Promotion', 'Returned', 'OrderID', 'CustomerName', 'ShippingCost', 'OrderDate', 'DeliveryDate', 'RegionManager']


,Date,Region,Product,Quantity,UnitPrice,StoreLocation,CustomerType,Discount,Salesperson,TotalPrice,PaymentMethod,Promotion,Returned,OrderID,CustomerName,ShippingCost,OrderDate,DeliveryDate,RegionManager
0,2023-02-23,East,Laptop,14,163.60,Store B,Wholesale,0.00,Eva,2290.400,Online,FREESHIP,0,REG100000,Cust 6583,43.34,2023-02-23,2023-02-27,Eric
1,2024-12-19,South,Phone,1,544.01,Store A,Retail,0.00,Alice,544.010,Gift Card,SAVE10,0,REG100001,Cust 2144,5.30,2024-12-19,2024-12-28,Sophie
2,2023-05-10,North,Desk,14,346.18,Store B,Wholesale,0.10,Alice,4361.868,Online,WINTER15,0,REG100002,Cust 5998,20.46,2023-05-10,2023-05-19,Ryan
3,2025-02-26,Central,Chair,18,384.82,Store A,Wholesale,0.15,Frank,5887.746,Gift Card,FREESHIP,0,REG100003,Cust 7136,27.95,2025-02-26,2025-03-02,Cameron
4,2023-06-24,East,Desk,18,237.76,Store C,Retail,0.00,Carlos,4279.680,Online,SAVE10,0,REG100004,Cust 6506,5.73,2023-06-24,2023-06-27,Eric


## 9. Save Raw Excel Copy

In [ ]:
shutil.copy2(input_path, raw_output_path)

print("Raw Excel saved:", to_relative_path(raw_output_path, PROJECT_ROOT))

Raw Excel saved: data/raw/excel/product_sales_region_raw.xlsx


## 10. Clean Column Names

In [ ]:
df_staging = df_raw.copy()

original_columns = df_staging.columns.tolist()
cleaned_columns = [clean_column_name(col) for col in original_columns]

df_staging.columns = cleaned_columns

print("Original columns:")
print(original_columns)

print("\nCleaned columns:")
print(cleaned_columns)

df_staging.head(5)

Original columns:
['Date', 'Region', 'Product', 'Quantity', 'UnitPrice', 'StoreLocation', 'CustomerType', 'Discount', 'Salesperson', 'TotalPrice', 'PaymentMethod', 'Promotion', 'Returned', 'OrderID', 'CustomerName', 'ShippingCost', 'OrderDate', 'DeliveryDate', 'RegionManager']

Cleaned columns:
['date', 'region', 'product', 'quantity', 'unitprice', 'storelocation', 'customertype', 'discount', 'salesperson', 'totalprice', 'paymentmethod', 'promotion', 'returned', 'orderid', 'customername', 'shippingcost', 'orderdate', 'deliverydate', 'regionmanager']


,date,region,product,quantity,unitprice,storelocation,customertype,discount,salesperson,totalprice,paymentmethod,promotion,returned,orderid,customername,shippingcost,orderdate,deliverydate,regionmanager
0,2023-02-23,East,Laptop,14,163.60,Store B,Wholesale,0.00,Eva,2290.400,Online,FREESHIP,0,REG100000,Cust 6583,43.34,2023-02-23,2023-02-27,Eric
1,2024-12-19,South,Phone,1,544.01,Store A,Retail,0.00,Alice,544.010,Gift Card,SAVE10,0,REG100001,Cust 2144,5.30,2024-12-19,2024-12-28,Sophie
2,2023-05-10,North,Desk,14,346.18,Store B,Wholesale,0.10,Alice,4361.868,Online,WINTER15,0,REG100002,Cust 5998,20.46,2023-05-10,2023-05-19,Ryan
3,2025-02-26,Central,Chair,18,384.82,Store A,Wholesale,0.15,Frank,5887.746,Gift Card,FREESHIP,0,REG100003,Cust 7136,27.95,2025-02-26,2025-03-02,Cameron
4,2023-06-24,East,Desk,18,237.76,Store C,Retail,0.00,Carlos,4279.680,Online,SAVE10,0,REG100004,Cust 6506,5.73,2023-06-24,2023-06-27,Eric


## 11. Define Required and Optional Fields

In [ ]:
required_fields = [
    "date",
    "region",
    "product",
    "quantity",
    "unitprice",
    "storelocation",
    "customertype",
    "discount",
    "salesperson",
    "totalprice",
    "paymentmethod",
    "returned",
    "orderid",
    "customername",
    "shippingcost",
    "orderdate",
    "deliverydate",
    "regionmanager",
]

optional_fields = [
    "promotion",
]

print("Required fields:", required_fields)
print("Optional fields:", optional_fields)

Required fields: ['date', 'region', 'product', 'quantity', 'unitprice', 'storelocation', 'customertype', 'discount', 'salesperson', 'totalprice', 'paymentmethod', 'returned', 'orderid', 'customername', 'shippingcost', 'orderdate', 'deliverydate', 'regionmanager']
Optional fields: ['promotion']


## 12. Validate Required Columns

In [ ]:
missing_required_columns = [
    col for col in required_fields
    if col not in df_staging.columns
]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

print("Required column validation passed.")

Required column validation passed.


## 13. Check Missing Values

In [ ]:
missing_values_all = df_staging.isna().sum()
required_missing_values = df_staging[required_fields].isna().sum()
optional_missing_values = df_staging[optional_fields].isna().sum()

total_missing_values = int(missing_values_all.sum())

print("All missing values:")
print(missing_values_all)

print("\nRequired missing values:")
print(required_missing_values)

print("\nOptional missing values:")
print(optional_missing_values)

print("\nTotal missing values:", total_missing_values)

All missing values:
date               0
region             0
product            0
quantity           0
unitprice          0
storelocation      0
customertype       0
discount           0
salesperson        0
totalprice         0
paymentmethod      0
promotion        370
returned           0
orderid            0
customername       0
shippingcost       0
orderdate          0
deliverydate       0
regionmanager      0
dtype: int64

Required missing values:
date             0
region           0
product          0
quantity         0
unitprice        0
storelocation    0
customertype     0
discount         0
salesperson      0
totalprice       0
paymentmethod    0
returned         0
orderid          0
customername     0
shippingcost     0
orderdate        0
deliverydate     0
regionmanager    0
dtype: int64

Optional missing values:
promotion    370
dtype: int64

Total missing values: 370


## 14. Save Parsed Output to Staging

In [ ]:
df_staging.to_csv(
    staging_output_path,
    index=False,
    encoding="utf-8"
)

print("Staging saved:", to_relative_path(staging_output_path, PROJECT_ROOT))

Staging saved: data/staging/excel/product_sales_region_staging.csv


## 15. Check Duplicate Rows

In [ ]:
duplicate_count = int(df_staging.duplicated().sum())

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


## 16. Create Clean Data

In [ ]:
df_clean = df_staging.copy()

df_clean = df_clean.dropna(subset=required_fields)
df_clean = df_clean.drop_duplicates()

records_valid = len(df_clean)
records_invalid = records_read - records_valid

print("Records read:", records_read)
print("Records valid:", records_valid)
print("Records invalid:", records_invalid)

df_clean.head(5)

Records read: 1500
Records valid: 1500
Records invalid: 0


,date,region,product,quantity,unitprice,storelocation,customertype,discount,salesperson,totalprice,paymentmethod,promotion,returned,orderid,customername,shippingcost,orderdate,deliverydate,regionmanager
0,2023-02-23,East,Laptop,14,163.60,Store B,Wholesale,0.00,Eva,2290.400,Online,FREESHIP,0,REG100000,Cust 6583,43.34,2023-02-23,2023-02-27,Eric
1,2024-12-19,South,Phone,1,544.01,Store A,Retail,0.00,Alice,544.010,Gift Card,SAVE10,0,REG100001,Cust 2144,5.30,2024-12-19,2024-12-28,Sophie
2,2023-05-10,North,Desk,14,346.18,Store B,Wholesale,0.10,Alice,4361.868,Online,WINTER15,0,REG100002,Cust 5998,20.46,2023-05-10,2023-05-19,Ryan
3,2025-02-26,Central,Chair,18,384.82,Store A,Wholesale,0.15,Frank,5887.746,Gift Card,FREESHIP,0,REG100003,Cust 7136,27.95,2025-02-26,2025-03-02,Cameron
4,2023-06-24,East,Desk,18,237.76,Store C,Retail,0.00,Carlos,4279.680,Online,SAVE10,0,REG100004,Cust 6506,5.73,2023-06-24,2023-06-27,Eric


## 17. Save Cleaned Output

In [ ]:
df_clean.to_csv(
    clean_output_path,
    index=False,
    encoding="utf-8"
)

print("Clean saved:", to_relative_path(clean_output_path, PROJECT_ROOT))

Clean saved: data/clean/excel/product_sales_region_clean.csv


## 18. Generate Ingestion Log

In [ ]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": to_relative_path(input_path, PROJECT_ROOT),
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "duplicate_rows_removed": int(duplicate_count),
    "sheet_names": sheet_names,
    "selected_sheet": selected_sheet,
    "required_fields": required_fields,
    "optional_fields": optional_fields,
    "missing_values": missing_values_all.astype(int).to_dict(),
    "required_missing_values": required_missing_values.astype(int).to_dict(),
    "optional_missing_values": optional_missing_values.astype(int).to_dict(),
    "total_missing_values": int(total_missing_values),
    "error_message": error_message,
    "raw_output_path": to_relative_path(raw_output_path, PROJECT_ROOT),
    "staging_output_path": to_relative_path(staging_output_path, PROJECT_ROOT),
    "clean_output_path": to_relative_path(clean_output_path, PROJECT_ROOT),
    "owner": owner
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("Log saved:", to_relative_path(log_output_path, PROJECT_ROOT))
ingestion_log

Log saved: logs/excel_ingestion_log.json


{'run_id': '5edff1eb-857e-4515-8a5d-31e2628466dc',
 'source_name': 'product_sales_region_excel',
 'source_type': 'excel',
 'input_path_or_url': 'data/sample_inputs/Product-Sales-Region.xlsx',
 'start_time': '2026-06-14T05:44:52.398162+00:00',
 'end_time': '2026-06-14T05:45:14.561673+00:00',
 'status': 'success',
 'records_read': 1500,
 'records_valid': 1500,
 'records_invalid': 0,
 'duplicate_rows_removed': 0,
 'sheet_names': ['Sheet1'],
 'selected_sheet': 'Sheet1',
 'required_fields': ['date',
  'region',
  'product',
  'quantity',
  'unitprice',
  'storelocation',
  'customertype',
  'discount',
  'salesperson',
  'totalprice',
  'paymentmethod',
  'returned',
  'orderid',
  'customername',
  'shippingcost',
  'orderdate',
  'deliverydate',
  'regionmanager'],
 'optional_fields': ['promotion'],
 'missing_values': {'date': 0,
  'region': 0,
  'product': 0,
  'quantity': 0,
  'unitprice': 0,
  'storelocation': 0,
  'customertype': 0,
  'discount': 0,
  'salesperson': 0,
  'totalprice':

## 19. Final Output Check

In [ ]:
print("Raw exists:", raw_output_path.exists())
print("Staging exists:", staging_output_path.exists())
print("Clean exists:", clean_output_path.exists())
print("Log exists:", log_output_path.exists())

print("\nOutput files:")
print(to_relative_path(raw_output_path, PROJECT_ROOT))
print(to_relative_path(staging_output_path, PROJECT_ROOT))
print(to_relative_path(clean_output_path, PROJECT_ROOT))
print(to_relative_path(log_output_path, PROJECT_ROOT))

Raw exists: True
Staging exists: True
Clean exists: True
Log exists: True

Output files:
data/raw/excel/product_sales_region_raw.xlsx
data/staging/excel/product_sales_region_staging.csv
data/clean/excel/product_sales_region_clean.csv
logs/excel_ingestion_log.json


## 20. Summary

```text
data/raw/excel/product_sales_region_raw.xlsx
data/staging/excel/product_sales_region_staging.csv
data/clean/excel/product_sales_region_clean.csv
logs/excel_ingestion_log.json
```